# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Run-Header

| Feld | Wert |
|---|---|
| Predictions | `daten/predictions.jsonl` (12 Anzeigen, generiert von `02_extract.ipynb`) |
| Gold | `annotation/meine_gold.csv` (Hand-Annotation aus Phase 2) |
| Modell | `Qwen/Qwen2.5-7B-Instruct` (Baseline, ohne Iteration) |
| Datum | 2026-05-18 |


## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Hypothese-Cell *vor* der Eval: welches Feld haltet ihr für am stärksten / am schwächsten — und warum?

## Hypothese vor der Eval

Vor Berechnung der konkreten Accuracy-Zahlen erwarte ich folgende Verteilung 
über die 6 Schema-Felder, basierend auf:
(a) der manuellen Inspektion einzelner Predictions in `02_extract.ipynb`,  
(b) der Korpus-Struktur (35 Festanstellung + 20 Ausbildung, viele HO-stumme Anzeigen),  
(c) der Allianz-Anzeige mit leeren Pflichtfeldern.

### Erwartung nach Feld

| Feld | Erwartete Stärke | Begründung |
|---|---|---|
| `vertragsart` | **stärkstes Feld** | Die Unterscheidung Ausbildung vs. Festanstellung ist im Text textuell sehr eindeutig (Wort "Ausbildung" steht explizit). Im κ aus Phase 2 hatten wir hier 100 % zwischen den menschlichen Annotator:innen — das Modell sollte dem nicht weit nachstehen. |
| `homeoffice` | stark | Im Korpus dominieren "stumme" Anzeigen → triviales `nicht_genannt`. Die wenigen `teilweise`/`ja`-Fälle haben sehr klare Signalwörter. Risiko: Modell überinterpretiert "flexible Arbeitszeit" als Homeoffice. |
| `gehalt_zeitraum` | stark | Wenn `gehalt_min_eur` null ist, muss auch der Zeitraum null sein — Konsistenz-Regel ist einfach. Risiko: nur die Wilken-Ausbildung hat eine echte Zahl. |
| `skills_top3` | mittel | Set-Match toleriert Reihenfolge, aber das Modell halluziniert manchmal nicht-technische Begriffe (z.B. "agile", "frameworks" — bereits in den Predictions gesichtet). |
| `erfahrungslevel` | **schwächstes Feld** | Schon in Phase 2 hatten zwei Menschen κ = 0.625 (substanziell, aber nicht super). Die Schema-Definition ist subjektiv ("mehrjährig", "routiniert" → mid oder senior?). Das Modell hat in Zelle-6-Test "mid" gewählt, wo Gold "nicht_genannt" war. |
| `gehalt_min_eur` | schwach | Das Modell hat bereits bei der Wilken-Ausbildung "Ausbildungsjahr: 953,00 €" übersehen (siehe `02_extract.ipynb` Befunde). Vermutlich Modell-Problem: Tausender-Format/Komma irritiert es. |

**Zusammengefasst — meine Prognose:**
- **Top 2 (höchste Accuracy):** `vertragsart`, `homeoffice`
- **Bottom 2 (niedrigste Accuracy):** `erfahrungslevel`, `gehalt_min_eur`

Nach Berechnung der echten Zahlen unten reflektiere ich, wo ich richtig und 
wo ich falsch lag.

In [1]:
"""
Imports + Daten einlesen.
"""
import json
import csv
from pathlib import Path
from collections import Counter

# Predictions laden
predictions = {}
with open("../daten/predictions.jsonl", encoding="utf-8") as f:
    for line in f:
        p = json.loads(line)
        predictions[p["refnr"]] = p

# Gold laden  
gold = {}
with open("../annotation/meine_gold.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if not row["id"].strip():
            continue
        gold[row["id"]] = row

# Gemeinsame IDs
gemeinsame_ids = sorted(set(predictions) & set(gold))

print(f"Predictions: {len(predictions)} Einträge")
print(f"Gold:        {len(gold)} Einträge")
print(f"Gemeinsam:   {len(gemeinsame_ids)} Einträge (für Eval verwendet)")

# Sanity: falls nicht 12 → was fehlt?
if len(gemeinsame_ids) != 12:
    nur_pred = set(predictions) - set(gold)
    nur_gold = set(gold) - set(predictions)
    if nur_pred: print(f"⚠ Nur in Predictions: {nur_pred}")
    if nur_gold: print(f"⚠ Nur in Gold: {nur_gold}")

Predictions: 12 Einträge
Gold:        12 Einträge
Gemeinsam:   12 Einträge (für Eval verwendet)


In [3]:
"""
Per-Field-Accuracy berechnen.

Konventionen:
- homeoffice, vertragsart, erfahrungslevel, gehalt_zeitraum: exakter String-Match
- gehalt_min_eur: exakter Match auf Integer (oder beide null/leer)
- skills_top3: SET-MATCH (Reihenfolge egal, Strings lowercase + gestripped)
- Leere/None-Werte in Predictions → Feld zählt als FALSCH
"""

# Welche Felder evaluieren wir?
KATEGORIAL = ["homeoffice", "vertragsart", "erfahrungslevel", "gehalt_zeitraum"]


def normalisiere_gold_wert(wert: str):
    """CSV-Gold: leerer String -> None, sonst gestrippt."""
    if wert is None:
        return None
    wert = wert.strip()
    return wert if wert else None


def gehalt_match(pred, gold_str):
    """gehalt_min_eur: int oder None."""
    gold_norm = normalisiere_gold_wert(gold_str)
    gold_int = int(gold_norm) if gold_norm and gold_norm.isdigit() else None
    
    if pred is None and gold_int is None:
        return True
    if pred is None or gold_int is None:
        return False
    return int(pred) == gold_int


def skills_match(pred_liste, gold_str):
    """Set-Match auf Skills (lowercase, gestripped)."""
    # Gold ist Pipe-getrennter String aus CSV: "python|sql|r"
    gold_norm = normalisiere_gold_wert(gold_str)
    if gold_norm:
        gold_set = {s.strip().lower() for s in gold_norm.split("|") if s.strip()}
    else:
        gold_set = set()
    
    # Predictions: Liste aus JSON
    if pred_liste is None:
        pred_set = set()
    elif isinstance(pred_liste, list):
        pred_set = {str(s).strip().lower() for s in pred_liste if s}
    else:
        pred_set = set()
    
    return pred_set == gold_set


# Pro Feld zählen
ergebnisse = {feld: {"korrekt": 0, "total": 0, "fehler": []} for feld in 
              KATEGORIAL + ["gehalt_min_eur", "skills_top3"]}

for refnr in gemeinsame_ids:
    p = predictions[refnr]
    g = gold[refnr]
    
    # Kategoriale Felder
    for feld in KATEGORIAL:
        pred_wert = p.get(feld)
        # Predictions: None oder leerer String -> als "leer" werten
        if pred_wert is None or pred_wert == "":
            pred_norm = None
        else:
            pred_norm = str(pred_wert).strip()
        
        gold_norm = normalisiere_gold_wert(g.get(feld, ""))
        
        ergebnisse[feld]["total"] += 1
        if pred_norm == gold_norm:
            ergebnisse[feld]["korrekt"] += 1
        else:
            ergebnisse[feld]["fehler"].append({
                "refnr": refnr, "pred": pred_norm, "gold": gold_norm
            })
    
    # gehalt_min_eur
    pred_geh = p.get("gehalt_min_eur")
    ergebnisse["gehalt_min_eur"]["total"] += 1
    if gehalt_match(pred_geh, g.get("gehalt_min_eur", "")):
        ergebnisse["gehalt_min_eur"]["korrekt"] += 1
    else:
        ergebnisse["gehalt_min_eur"]["fehler"].append({
            "refnr": refnr, "pred": pred_geh, "gold": g.get("gehalt_min_eur", "")
        })
    
    # skills_top3
    pred_sk = p.get("skills_top3")
    ergebnisse["skills_top3"]["total"] += 1
    if skills_match(pred_sk, g.get("skills_top3", "")):
        ergebnisse["skills_top3"]["korrekt"] += 1
    else:
        ergebnisse["skills_top3"]["fehler"].append({
            "refnr": refnr, "pred": pred_sk, "gold": g.get("skills_top3", "")
        })


# Tabelle ausgeben
print(f"{'Feld':<20} {'Accuracy':>10} {'n_korrekt':>12} {'n_total':>10}")
print("─" * 56)
for feld in ["homeoffice", "vertragsart", "erfahrungslevel", 
             "gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]:
    e = ergebnisse[feld]
    acc = e["korrekt"] / e["total"] if e["total"] else 0
    print(f"{feld:<20} {acc:>9.1%} {e['korrekt']:>12} {e['total']:>10}")

# Gesamtaccuracy
total_korrekt = sum(e["korrekt"] for e in ergebnisse.values())
total_zellen = sum(e["total"] for e in ergebnisse.values())
print("─" * 56)
print(f"{'GESAMT':<20} {total_korrekt/total_zellen:>9.1%} {total_korrekt:>12} {total_zellen:>10}")

# Hinweis auf leere Predictions
parse_fails = sum(
    1 for refnr in gemeinsame_ids 
    if predictions[refnr].get("parse_fail") 
    or not predictions[refnr].get("homeoffice")
)
print(f"\nHinweis: {parse_fails} Anzeige(n) mit fehlenden Pflichtfeldern "
      f"(zählen über alle Felder als Fail)")

Feld                   Accuracy    n_korrekt    n_total
────────────────────────────────────────────────────────
homeoffice               75.0%            9         12
vertragsart              91.7%           11         12
erfahrungslevel          66.7%            8         12
gehalt_min_eur           91.7%           11         12
gehalt_zeitraum          91.7%           11         12
skills_top3               8.3%            1         12
────────────────────────────────────────────────────────
GESAMT                   70.8%           51         72

Hinweis: 1 Anzeige(n) mit fehlenden Pflichtfeldern (zählen über alle Felder als Fail)


In [4]:
"""
Konkrete Fehlerfälle anschauen — die zwei schwächsten Felder.
Pflicht laut Aufgabenblatt: Hypothesen an Anzeigen begründen.
"""

print("="*80)
print("SCHWÄCHSTES FELD 1: skills_top3 (8.3% Accuracy)")
print("="*80)
for fall in ergebnisse["skills_top3"]["fehler"]:
    refnr = fall["refnr"]
    print(f"\nrefnr: {refnr}")
    print(f"  Pred:  {fall['pred']}")
    print(f"  Gold:  {fall['gold']}")

print("\n")
print("="*80)
print("SCHWÄCHSTES FELD 2: erfahrungslevel (66.7% Accuracy)")
print("="*80)
for fall in ergebnisse["erfahrungslevel"]["fehler"]:
    refnr = fall["refnr"]
    print(f"\nrefnr: {refnr}")
    print(f"  Pred:  {fall['pred']!r}")
    print(f"  Gold:  {fall['gold']!r}")

SCHWÄCHSTES FELD 1: skills_top3 (8.3% Accuracy)

refnr: 10000-1185445644-S
  Pred:  ['python', 'r', 'sql']
  Gold:  python|spark|ci/cd

refnr: 10001-1002026714-S
  Pred:  ['java', 'c#', 'sql']
  Gold:  

refnr: 10001-1002616378-S
  Pred:  ['python', 'pandas', 'scikit-learn']
  Gold:  python|machine learning|azure

refnr: 10001-1002678126-S
  Pred:  ['python', 'machine-learning', 'scikit-learn']
  Gold:  python|machine learning|git

refnr: 10001-1003030981-S
  Pred:  ['java', 'frameworks', 'testen']
  Gold:  

refnr: 13319-868490/1_607417LS-S
  Pred:  ['python', 'machine-learning', 'sql']
  Gold:  python|machine learning|sql

refnr: 13644-297872-S
  Pred:  ['python', 'pandas', 'scikit-learn']
  Gold:  python|machine learning|pandas

refnr: 15719-44156922-55-S
  Pred:  None
  Gold:  python|machine learning|SQL

refnr: 16818-100787927-S
  Pred:  ['java', 'delphi', 'sql']
  Gold:  

refnr: 16818-100814418-S
  Pred:  ['java', 'ms-office', 'agile']
  Gold:  java|ms office

refnr: 19384-93844

## Reflexion + Diagnose

### Hypothese-Reflexion (vs. Vorhersage oben)

| Feld | Vorhersage | Tatsächlich | Bewertung |
|---|---|---|---|
| `vertragsart` | stärkstes | **91.7 %** | ✅ wie erwartet (Top 1) |
| `homeoffice` | stark | 75.0 % | ⚠️ schwächer als erwartet (z.B. Noweda-HO übersehen) |
| `gehalt_zeitraum` | stark | **91.7 %** | ✅ Konsistenzregel funktioniert |
| `skills_top3` | mittel | **8.3 %** | ❌ **dramatisch unterschätzt** — siehe Klassifikation unten |
| `erfahrungslevel` | schwächstes | 66.7 % | ✅ wie erwartet (Bottom 2) |
| `gehalt_min_eur` | schwach | **91.7 %** | ❌ überschätzt — das Modell macht "null bei Unsicherheit", was zu hoher Accuracy führt |

**Lehre:** Mein größter Fehler war die Skills-Einschätzung. Ich hatte das Set-Match-Verfahren als großzügig gesehen, aber **`machine-learning` ≠ `machine learning`** killt allein 3 Treffer. Das ist keine inhaltliche Schwäche des Modells, sondern eine Pipeline-Schwäche.

### Zwei schwächste Felder + Iterationskandidaten für Phase 4

#### 1. `skills_top3` — Accuracy 8.3 % (1 / 12)

Die 11 Fehler verteilen sich auf vier klar trennbare Klassen:

- **Schreibweise-Varianten (3 Fälle):** `machine-learning` vs. `machine learning`, `ms-office` vs. `ms office`. Inhaltlich identisch, aber Set-Match-Pipeline wertet als Disagreement. → **Pipeline-Problem**, lässt sich durch Normalisierung beheben (z.B. Bindestriche → Leerzeichen vor Set-Vergleich).
- **Ausbildung "wird gelernt"-Skills (3 Fälle):** Bei Atruvia, msg, Wilken, DS Datentechnik extrahiert das Modell `java`, `delphi`, `c#`, obwohl diese im Text als Lerngegenstand, nicht als Anforderung stehen. Ich habe sie als Annotator weggelassen. → **Schema-Problem**, das Schema definiert nicht, ob "wird in der Ausbildung vermittelt" als Skill zählt.
- **Halluzinierte Nicht-Tool-Begriffe (1 Fall):** Atruvia mit `agile`, `frameworks`, `testen`. Methoden/Konzepte, keine konkreten Tools. → **Modell-Problem** (Prompt müsste expliziter sagen: nur konkrete Tools/Sprachen, keine Methoden).
- **Echtes Skill-Disagreement (4 Fälle):** Beide wählen 3 von 5–6 möglichen Skills, aber unterschiedliche. Beispiel Noweda: ich hatte `r|sql|machine learning`, Modell `python|r|sql`. Python ist im Anzeigentext gar nicht erwähnt → Modell halluziniert vermutlich aus Branchenwissen. → teils **Modell-Problem** (Halluzination), teils inhärente Unschärfe des Schemas (keine eindeutige Top-3-Auswahl-Regel).

**Iterations-Vermutung für Phase 4:** Skills-Pipeline normalisieren + Prompt schärfen ("Nenne nur konkrete im Text genannte Tools, keine Methoden") → Accuracy sollte deutlich steigen.

#### 2. `erfahrungslevel` — Accuracy 66.7 % (8 / 12)

Die 4 Fehler:

- **Peagle (refnr 10001-1002678126-S):** Modell `mid`, Gold `junior`. Im Text steht eindeutig *"Gerne erste Berufserfahrung nach dem Studium (maximal 2 Jahre)"*. Das ist die klarste Junior-Definition möglich. → **Modell-Problem** (übergewichtet das Wort "Berufserfahrung", ignoriert "maximal 2 Jahre").
- **KOSATEC (refnr 13644-294294-S):** Modell `mid`, Gold `nicht_genannt`. Genau der Edge Case aus Phase 2 (Annotator A vs. Annotator B). → **Schema-Problem** (siehe Phase-2-Edge-Cases).
- **Allianz (refnr 15719-44156922-55-S):** Modell `None`, Gold `nicht_genannt`. Truncation hat den Profil-Block abgeschnitten. → **Pipeline-Problem** (bereits in `02_extract.ipynb` diagnostiziert).
- **Noweda (refnr 19384-938440348-S):** Modell `nicht_genannt`, Gold `junior`. Im Text *"Erste Berufserfahrung in Data-Science-Projekten ist erwünscht, jedoch nicht zwingend erforderlich"* → klare Junior-Anzeige. → **Modell-Problem** (interpretiert "nicht zwingend" als "nicht_genannt").

**Iterations-Vermutung für Phase 4:** Im Prompt explizite Beispiele für "≤2 Jahre / erste Berufserfahrung" → junior. Zusätzlich Truncation-Strategie überdenken (Allianz fängt sich dann mit).

### Plan für Phase 4

Iteration A wird **eine** der beiden Schwächen gezielt angehen — ich entscheide dort (mit Hypothese vorab), welche Aktion erwartbar den größten Hebel hat.

## Phase 4 — Iteration A Auswertung

Hypothese-Cell *vor* der Iteration: welches Feld, welche Δ-Größe, warum?

## Phase 4 — Iteration B Auswertung

Hypothese-Cell *vor* der Iteration.

## Phase 4 — Synthese

Iterations-Tabelle (Baseline / A / B mit Hypothese, Aktion, Δ Gesamt, Δ schwächstes Feld, Diagnose) + Synthese-Antworten zu den drei Fragen aus dem Aufgabenblatt.

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Per-Field-Accuracy auf den 12 Hand-Gold-Anzeigen + Schema-Konformitäts-Check auf den restlichen Anzeigen ohne Gold. 3B-vs-7B-vs-Gold per `refnr` joinen, drei eigenständige Halluzinations-Klassen mit konkreten Beispielen identifizieren.